# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore a Croissant-format dataset using the [`mlcroissant`](https://mlcommons.org/croissant) library.

The dataset reports on mental health indicators among residents of Kilifi County, Kenya, including demographics and standardized scale results (GAD-7, PHQ-9, PSQ), and is expressed using AI-ready FAIR² data standards.

### Dataset Source
The dataset source is provided via a Croissant schema JSON-LD URL:

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant metadata URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields by their @id

print("Available Record Sets (@id and name):\n")
for rs in metadata.record_sets:
    print(f"- @id: {rs.id}  | name: {rs.name}")

# For each record set, print its fields and columns
for rs in metadata.record_sets:
    print(f"\nRecord Set @id={rs.id} (name={rs.name}) fields and columns:")
    for field in rs.fields:
        print(f"  - Field @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
        # Print columns for this field if present
        if field.columns:
            print(f"    Columns for this field:")
            for col in field.columns:
                print(f"      - Column @id: {col.id} | name: {col.name} | dataType: {col.data_type}")

## 3. Data Extraction
Load data from available record sets into DataFrames for analysis.
All entities (record set, field, column) are referenced by their `@id`s, as shown above.

In [ ]:
# Choose all record set @ids for loading
record_set_ids = [rs.id for rs in metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in @id {record_set_id}: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print(f"No records found for {record_set_id}.")

# For further analysis we'll use the first loaded record set
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    main_df = dataframes[main_record_set_id]
    print(f"Using main record set @id: {main_record_set_id} for EDA.")
else:
    main_record_set_id = None
    main_df = pd.DataFrame()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by demographic attributes. All fields and columns are referenced by their `@id`. For illustration, let's analyze the GAD-7 score field (commonly present in such surveys), using its `@id`.

In [ ]:
#### --- Customize these IDs based on prior overview --- ####

# Example field IDs for analysis, update based on printed overview above
# (Fields may be named e.g., 'gad7_score', '@id': 'gad7_score', or similar)

# If you don't know the field @id, check the result from previous overview
# For this example, we suppose field with id 'gad7_score' (edit as needed)
numeric_field_id = 'gad7_score'  # e.g., '@id' of GAD-7 total score field

if numeric_field_id in main_df.columns:
    # Convert to numeric in case
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    
    # Filter records with GAD-7 score > threshold
    threshold = 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:\n")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by education level or other group field. Use @id from overview
    group_field_id = 'level_of_education'   # Example @id for education field
    if group_field_id in main_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
    else:
        print(f"Group field {group_field_id} not found in columns.")
else:
    print(f"Column {numeric_field_id} not found in DataFrame.")

## 5. Visualization
Visualize data distributions or relationships between key fields. Here, we'll plot the distribution of GAD-7 scores and display mean GAD-7 by education, if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df.shape[0] > 0 and numeric_field_id in main_df.columns:
    plt.figure(figsize=(7,4))
    main_df[numeric_field_id].dropna().hist(bins=14)
    plt.title(f"Distribution of GAD-7 Scores (@id={numeric_field_id})")
    plt.xlabel("GAD-7 score")
    plt.ylabel("Count")
    plt.show()

    # If group field exists, display mean per group as barplot
    if group_field_id in main_df.columns:
        plt.figure(figsize=(9,4))
        means = main_df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
        means.plot(kind='bar')
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean GAD-7 Score by Education Level")
        plt.tight_layout()
        plt.show()
else:
    print("Not enough data for plotting.")

## 6. Conclusion
We successfully loaded and explored the FAIR² mental health dataset for Kilifi County, Kenya, via the `mlcroissant` library, referencing all dataset entities using their `@id` as required by Croissant data standards. We loaded record sets, examined field and column structures, and conducted basic EDA and visualizations of summary data (e.g. GAD-7 scores).

**For deeper insights, consult the dataset's full Croissant metadata and documentation, and tailor further analysis to specific research questions.**